In [ ]:
from datasets import load_dataset

ds = load_dataset("Genius-Society/Pima")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [ ]:
# Check dataset structure
print(ds)

# View first sample
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['ID', 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
        num_rows: 614
    })
    validation: Dataset({
        features: ['ID', 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
        num_rows: 77
    })
    test: Dataset({
        features: ['ID', 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
        num_rows: 77
    })
})
{'ID': 235, 'Pregnancies': 4, 'Glucose': 171, 'BloodPressure': 72, 'SkinThickness': 29, 'Insulin': 155, 'BMI': 43.6, 'DiabetesPedigreeFunction': 0.479, 'Age': 26, 'Outcome': 1}


In [ ]:
!pip install pennylane datasets

import pennylane as qml
from pennylane import numpy as np
# Extract features and labels into arrays
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'], s['SkinThickness'],
               s['Insulin'], s['BMI'], s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']])
y = np.array([s['Outcome'] for s in ds['train']])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 86.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [ ]:
print(X.shape)
print(y.shape)

(614, 8)
(614,)


In [ ]:
# Simulateur quantique
num_qubits = X.shape[1]  # 8 features
dev = qml.device("default.qubit", wires=num_qubits)

# Normalisation
X_norm = X / X.max(axis=0)  # valeurs entre 0 et 1

# QNode Angle Encoding Rx + Ry
@qml.qnode(dev)
def rx_ry_encoding(x_norm):
    for i in range(len(x_norm)):
        theta_x = x_norm[i] * np.pi      # angle pour Rx
        theta_y = x_norm[i] * (np.pi/2)  # angle pour Ry
        qml.RX(theta_x, wires=i)
        qml.RY(theta_y, wires=i)
    return qml.state()

# Exemple : premier patient
state_patient1 = rx_ry_encoding(X_norm[0])
print("État quantique du premier patient :", state_patient1)

État quantique du premier patient : [-1.14262010e-01-4.64232861e-02j -5.01459188e-02+6.85733223e-02j
 -3.12628157e-02+3.10153028e-02j  1.54895130e-02+2.60805136e-02j
  4.89245012e-02+1.55538970e-01j  1.12114374e-01-6.65128067e-03j
  5.79366301e-02+5.73742019e-03j  1.35323281e-02-3.77502037e-02j
 -2.90824988e-02+2.83436119e-02j  1.40694027e-02+2.41764706e-02j
  5.23101766e-03+1.35238527e-02j  9.91172998e-03-1.23118189e-03j
  5.33777061e-02+5.76566341e-03j  1.27880156e-02-3.46994060e-02j
  9.37608439e-03-1.67207237e-02j -9.60244520e-03-9.06380213e-03j
 -5.07341091e-02+1.00212805e-01j  5.84646198e-02+5.06743215e-02j
  2.57973076e-02+3.07090031e-02j  2.48369660e-02-1.20957919e-02j
  1.44852516e-01-3.26990592e-02j  2.40087803e-03-1.02257953e-01j
  9.56771136e-03-5.21527139e-02j -3.32444417e-02-1.51233055e-02j
  2.35365150e-02+2.85290300e-02j  2.30019388e-02-1.09501679e-02j
  1.26682210e-02-3.72997799e-03j -3.71490526e-04-9.08875460e-03j
  9.25024116e-03-4.80128124e-02j -3.05314900e-02-1.421

In [ ]:
# 4. Transformation de TOUS les patients (Phase Quantique)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
print("Encodage quantique en cours... (Patient par patient)")
# On génère les états pour chaque ligne de X_norm
all_states = np.array([rx_ry_encoding(x) for x in X_norm])

# 5. Conversion pour l'Arbre de Décision (Phase Classique)
# IMPORTANT : On extrait la partie réelle car l'arbre ne gère pas les complexes
X_final = np.real(all_states)

# 6. Découpage et Entraînement de l'Arbre de Décision
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

print("Entraînement de l'Arbre de Décision...")
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# 7. Résultat
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("-" * 30)
print(f"RÉSULTAT ANGLE ENCODING (8 Qubits)")
print(f"Précision (Accuracy) : {acc * 100:.2f}%")
print("-" * 30)

Encodage quantique en cours... (Patient par patient)
Entraînement de l'Arbre de Décision...
------------------------------
RÉSULTAT ANGLE ENCODING (8 Qubits)
Précision (Accuracy) : 73.17%
------------------------------
